# BigSmall Quickstart

**Run any model. No compromises.**

BigSmall is a *lossless* neural-network weight compressor. Mistral 7B goes from 14 GB to 9.3 GB. After decompression every weight is bit-for-bit identical to the original (md5-verified). No quantization, no quality loss.

This notebook walks through the full workflow end-to-end:

1. Installation
2. Compress a local `.safetensors` file
3. Decompress and verify
4. HuggingFace integration — `compress_for_hub`, `upload_to_hub`, `from_pretrained`
5. Streaming loader — load Mistral 7B with peak RAM under 2 GB
6. Fine-tune delta compression — ship deltas as 6.95% of the source size
7. CLI usage examples

Everything below is runnable on a machine with a GPU and the optional `[hf]` extras installed.

## 1. Installation

Install BigSmall from PyPI. The base install contains the codecs, container, encoder/decoder, and CLI. Add `[hf]` for HuggingFace Hub integration, `[diffusion]` for Stable Diffusion, or `[all]` for everything.

In [ ]:
!pip install bigsmall
# Optional extras:
# !pip install "bigsmall[hf]"
# !pip install "bigsmall[diffusion]"
# !pip install "bigsmall[all]"

In [ ]:
import bigsmall
from pathlib import Path

print("bigsmall", bigsmall.__version__ if hasattr(bigsmall, "__version__") else "(installed)")

## 2. Compress a local safetensors file

`bigsmall.compress(src, dst)` takes a `.safetensors` file and writes a `.bs` container. It auto-detects the float format (BF16, FP16, FP32, FP8, FP4) and picks the matching joint-entropy codec.

Below we use GPT-2 as a small, fast example. Swap in any safetensors path you have locally.

In [ ]:
# Download GPT-2 weights once, then compress them locally.
from huggingface_hub import hf_hub_download

src = Path(hf_hub_download("gpt2", "model.safetensors"))
dst = Path("gpt2.bs")

bigsmall.compress(src, dst)
print(f"original:   {src.stat().st_size / 1e6:.1f} MB")
print(f"compressed: {dst.stat().st_size / 1e6:.1f} MB")
print(f"ratio:      {dst.stat().st_size / src.stat().st_size * 100:.1f}%")

### Quick look at the container

`bigsmall.info()` reads only the header — no decompression — and reports the format, codec, tensor count, and the achieved ratio.

In [ ]:
info = bigsmall.info(dst)
for k, v in info.items():
    print(f"{k:>20}: {v}")

## 3. Decompress and verify

`bigsmall.decompress(src, dst)` reconstructs the safetensors file. `bigsmall.verify(bs_path, original_safetensors)` checks every tensor against the md5 stored in the container — if anything drifted by even one bit, it raises.

In [ ]:
restored = Path("gpt2_restored.safetensors")
bigsmall.decompress(dst, restored)

bigsmall.verify(dst, src)  # raises if any tensor's md5 differs
print("OK — bit-identical reconstruction verified")

In [ ]:
# Loading straight to torch tensors (no intermediate file) is also one call:
import bigsmall
state_dict = bigsmall.load(dst, device="cpu")
print(f"{len(state_dict)} tensors loaded")
print("first key:", next(iter(state_dict)))

## 4. HuggingFace integration

Three calls cover the full round-trip:

- `compress_for_hub(source, output_dir)` — compress any HF model (local dir *or* repo ID) into shard-aware `.bs` files plus a `bigsmall.index.json`.
- `upload_to_hub(output_dir, repo_id)` — push the compressed shards and config files to a Hub repo (created if missing).
- `from_pretrained(repo_or_path)` — download + decompress to a torch `state_dict` that drops directly into `model.load_state_dict(...)`.

In [ ]:
# Compress an HF model (repo ID or local path)
bigsmall.compress_for_hub("gpt2", output_dir="./gpt2_bs")
list(Path("./gpt2_bs").iterdir())

In [ ]:
# Upload to the Hub (requires `huggingface-cli login` or HF_TOKEN env var).
# bigsmall.upload_to_hub("./gpt2_bs", "your-username/gpt2-bigsmall")

In [ ]:
# Load a pre-compressed model directly from the Hub.
state_dict = bigsmall.from_pretrained("wpferrell/gpt2-bigsmall", device="cpu")
print(f"{len(state_dict)} tensors loaded from Hub")

# from transformers import GPT2LMHeadModel, GPT2Config
# model = GPT2LMHeadModel(GPT2Config())
# model.load_state_dict(state_dict, strict=False)

## 5. Streaming loader

The streaming loader decompresses **one transformer layer at a time**, directly into the target device. The previous layer is freed before the next is decoded, so peak memory is `embeddings + one layer` — typically under 2 GB even for a 7B model.

Below we stream Mistral 7B Instruct on a machine that doesn't have 14 GB free.

In [ ]:
with bigsmall.StreamingLoader("wpferrell/mistral-7b-instruct-bigsmall", device="cuda") as loader:
    print(f"{loader.layer_count()} transformer layers")
    print(f"{len(loader.non_layer_tensor_names())} non-layer tensors (embeddings, norms, lm_head)")

    # Pull embeddings + norms upfront (small).
    base = loader.load_non_layer_tensors()

    # Stream layers one at a time. Peak RAM stays under ~2 GB.
    for layer_idx, layer_tensors in loader.iter_layers():
        # `layer_tensors` is a dict[str, torch.Tensor] on `device`.
        # Previous layer is already freed (CUDA cache emptied between iters).
        if layer_idx < 3:
            sample = next(iter(layer_tensors))
            print(f"  layer {layer_idx:>2}: {sample} {tuple(layer_tensors[sample].shape)}")
        # ... run this layer's forward pass here, then drop the reference ...

If you want a `torch.nn.Module` that transparently streams layers on every forward pass, see `bigsmall.streaming_model` for the higher-level wrapper.

## 6. Fine-tune delta compression

When you fine-tune a base model, the resulting weights are *almost* the base. BigSmall XORs the fine-tune against the base on a per-tensor basis — the XOR is mostly zero-bytes and compresses far better. In practice a Mistral 7B fine-tune ships as **6.95% of the source size**, so you can publish 30 fine-tunes for the cost of one full model.

```python
import bigsmall

# Compress a fine-tune as a delta against its base
bigsmall.compress_delta(
    "mistral_finetune.safetensors",     # the fine-tuned weights
    "mistral_base.safetensors",         # the original base
    "finetune_delta.bs",
)

# Reconstruct the full fine-tuned weights from delta + base
bigsmall.decompress_delta(
    "finetune_delta.bs",
    "mistral_base.safetensors",
    "mistral_finetune_restored.safetensors",
)
```

The base can be either a `.safetensors` or a `.bs` file.

## 7. CLI usage

Everything above is exposed as the `bigsmall` console script. The CLI is the easiest way to script compression in CI pipelines.

In [ ]:
!bigsmall --help

```bash
# Compress (default: balanced)
bigsmall compress model.safetensors                # -> model.bs
bigsmall compress model.safetensors --storage      # smallest output
bigsmall compress model.safetensors --inference    # fastest decode

# Inspect / verify
bigsmall info model.bs
bigsmall verify model.bs

# Decompress
bigsmall decompress model.bs -o model.safetensors

# Fine-tune delta
bigsmall compress finetune.safetensors --base base.safetensors -o delta.bs
bigsmall decompress delta.bs --base base.safetensors -o reconstructed.safetensors
```

---

**Next steps:**

- Pre-compressed models on the Hub: [`wpferrell/mistral-7b-instruct-bigsmall`](https://huggingface.co/wpferrell/mistral-7b-instruct-bigsmall), [`wpferrell/gpt2-bigsmall`](https://huggingface.co/wpferrell/gpt2-bigsmall)
- Source: [github.com/wpferrell/Bigsmall](https://github.com/wpferrell/Bigsmall)
- License: Apache 2.0